# Notebook 05 — Evaluation & Grad-CAM

**Self-contained** — all functions inline.

Gold evaluation: builds test_ds via `from_tensor_slices` for guaranteed alignment.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, auc,
                              f1_score, accuracy_score)
from PIL import Image

print(f'TensorFlow: {tf.__version__}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────
DATA_DIR    = 'processed_data'
MODEL_PATH  = 'models/nsclc_stage_classifier.keras'
METRICS_DIR = 'metrics'
FIGURES_DIR = 'figures'
GRADCAM_DIR = os.path.join(FIGURES_DIR, 'gradcam')

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 16
NUM_CLASSES = 3
ALL_LABELS  = [0, 1, 2]
STAGE_NAMES = ['Stage I', 'Stage II', 'Stage III']

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(GRADCAM_DIR, exist_ok=True)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS (inline)
# ══════════════════════════════════════════════════════════════════

def detect_collapse(y_prob, y_pred=None, threshold=0.9, class_names=None):
    if y_pred is None:
        y_pred = y_prob.argmax(axis=1)
    n = len(y_pred)
    unique, counts = np.unique(y_pred, return_counts=True)
    print(f'  Mean softmax: {y_prob.mean(axis=0)}')
    print(f'  Std softmax:  {y_prob.std(axis=0)}')
    for cls, cnt in zip(unique, counts):
        name = class_names[cls] if class_names else f'Class {cls}'
        print(f'    {name}: {cnt}/{n} ({cnt/n*100:.1f}%)')
    max_frac = counts.max() / n
    if max_frac >= threshold:
        dom = unique[counts.argmax()]
        name = class_names[dom] if class_names else f'Class {dom}'
        print(f'  ⚠️ COLLAPSE: {name} = {max_frac*100:.1f}%')
        return True
    print(f'  ✅ No collapse (max={max_frac*100:.1f}%)')
    return False


def plot_roc(y_true, y_prob, class_names, save_path):
    from sklearn.preprocessing import label_binarize
    n_classes = len(class_names)
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(8, 6))
    for i in range(n_classes):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{class_names[i]} (AUC={roc_auc:.3f})')
    ax.plot([0,1], [0,1], 'k--', alpha=0.3)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curves'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); fig.savefig(save_path, dpi=150); plt.close(fig)
    print(f'Saved {save_path}')


def generate_gradcam(model, image, layer_name=None):
    """Generate Grad-CAM heatmap."""
    if layer_name is None:
        for layer in reversed(model.layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                layer_name = layer.name
                break
            if isinstance(layer, tf.keras.Model):
                for sub in reversed(layer.layers):
                    if isinstance(sub, tf.keras.layers.Conv2D):
                        layer_name = sub.name
                        break
                if layer_name:
                    break
    
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    
    img_tensor = tf.expand_dims(tf.constant(image, dtype=tf.float32), 0)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tensor)
        pred_class = tf.argmax(preds[0])
        class_channel = preds[:, pred_class]
    
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def save_gradcam_overlay(image, heatmap, save_path, title=''):
    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis], image.shape[:2]).numpy()[:,:,0]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image[:,:,0], cmap='gray'); axes[0].set_title('Original')
    axes[1].imshow(heatmap_resized, cmap='jet'); axes[1].set_title('Heatmap')
    axes[2].imshow(image[:,:,0], cmap='gray')
    axes[2].imshow(heatmap_resized, cmap='jet', alpha=0.4)
    axes[2].set_title('Overlay')
    for ax in axes: ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=11)
    plt.tight_layout(); fig.savefig(save_path, dpi=100); plt.close(fig)


print('✅ Utility functions defined')

In [ ]:
# ── Verify labels ────────────────────────────────────────────────
for name in ['train_slices', 'val_slices', 'test_slices']:
    df = pd.read_csv(os.path.join(DATA_DIR, f'{name}.csv'))
    dist = dict(df['label'].value_counts().sort_index())
    pts = dict(df.groupby('label')['patient_id'].nunique())
    print(f'{name}: {dist} ({pts} patients)')

In [ ]:
# ── Load model ───────────────────────────────────────────────────
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f'Model: {MODEL_PATH}, output={model.output_shape}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# GOLD EVALUATION — guaranteed alignment via from_tensor_slices
# ══════════════════════════════════════════════════════════════════
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_slices.csv'))

paths = [os.path.join(DATA_DIR, p) for p in test_df['path_image'].tolist()]
labels = test_df['label'].astype('int32').to_numpy()
patient_ids = test_df['patient_id'].astype(str).to_numpy()

def _load(path, label, pid):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.repeat(img, 3, axis=-1)
    return img, label, pid

test_ds = (
    tf.data.Dataset.from_tensor_slices((paths, labels, patient_ids))
    .map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f'Test: {len(test_df)} slices, {test_df["patient_id"].nunique()} patients')

In [ ]:
# ── Predict (aligned) ────────────────────────────────────────────
y_true_l, y_prob_l, pids_l = [], [], []

for xb, yb, pb in test_ds:
    probs = model.predict(xb, verbose=0)
    y_prob_l.append(probs)
    y_true_l.append(yb.numpy())
    pids_l.append(pb.numpy())

y_prob = np.vstack(y_prob_l)
y_pred = y_prob.argmax(axis=1)
y_true = np.concatenate(y_true_l)
pids   = np.concatenate(pids_l).astype(str)

assert len(y_true) == len(test_df)
print(f'y_true dist: {dict(zip(*np.unique(y_true, return_counts=True)))}')
print(f'y_pred dist: {dict(zip(*np.unique(y_pred, return_counts=True)))}')

print('\n── Collapse Detection ──')
detect_collapse(y_prob, y_pred, class_names=STAGE_NAMES)

print('\nSample predictions:')
for i in range(min(8, len(y_prob))):
    p = ', '.join([f'{v:.3f}' for v in y_prob[i]])
    print(f'  True={y_true[i]} Pred={y_pred[i]} [{p}]')

In [ ]:
# ═════════════════════════════════════════════════════════════════
# SLICE-LEVEL EVALUATION
# ═════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('SLICE-LEVEL EVALUATION')
print('='*60)

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f'Accuracy: {acc:.4f}')
print(f'Macro F1: {f1_macro:.4f}')
print(f'Weighted F1: {f1_weighted:.4f}')

print('\n' + classification_report(
    y_true, y_pred, labels=ALL_LABELS,
    target_names=STAGE_NAMES, zero_division=0))

# Save metrics
metrics = {'accuracy': float(acc), 'macro_f1': float(f1_macro),
           'weighted_f1': float(f1_weighted)}
with open(os.path.join(METRICS_DIR, 'slice_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

In [ ]:
# ── Confusion matrices ───────────────────────────────────────────
for norm, sfx in [(None, 'raw'), ('true', 'normalized')]:
    cm = confusion_matrix(y_true, y_pred, labels=ALL_LABELS, normalize=norm)
    fig, ax = plt.subplots(figsize=(8, 6))
    fmt = '.2f' if norm else 'd'
    ConfusionMatrixDisplay(cm, display_labels=STAGE_NAMES).plot(
        ax=ax, cmap='Blues', values_format=fmt)
    ax.set_title(f'Slice Confusion Matrix ({sfx})')
    plt.tight_layout()
    p = os.path.join(FIGURES_DIR, f'slice_cm_{sfx}.png')
    fig.savefig(p, dpi=150); plt.close(fig)
    print(f'Saved {p}')

plot_roc(y_true, y_prob, STAGE_NAMES,
         os.path.join(FIGURES_DIR, 'slice_roc.png'))

In [ ]:
# ═════════════════════════════════════════════════════════════════
# PATIENT-LEVEL
# ═════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('PATIENT-LEVEL EVALUATION')
print('='*60)

def patient_agg(pids, y_prob, y_true, method='mean'):
    pid_idx = {}
    for i, pid in enumerate(pids):
        pid_idx.setdefault(pid, []).append(i)
    uids, pt_probs, pt_labels = [], [], []
    for pid, idxs in pid_idx.items():
        uids.append(pid)
        sp = y_prob[idxs]
        agg = sp.mean(0) if method == 'mean' else sp.max(0)
        agg = agg / max(agg.sum(), 1e-7)
        pt_probs.append(agg)
        pt_labels.append(y_true[idxs[0]])
    pt_probs = np.array(pt_probs)
    return np.array(uids), pt_probs, pt_probs.argmax(1), np.array(pt_labels, dtype=np.int32)

for method in ['mean', 'max']:
    print(f'\n--- {method} pooling ---')
    pt_ids, pt_probs, pt_preds, pt_labels = patient_agg(pids, y_prob, y_true, method)
    detect_collapse(pt_probs, pt_preds, class_names=STAGE_NAMES)
    
    pt_acc = accuracy_score(pt_labels, pt_preds)
    pt_f1 = f1_score(pt_labels, pt_preds, average='macro', zero_division=0)
    print(f'  Patients: {len(pt_ids)}, Acc: {pt_acc:.4f}, Macro F1: {pt_f1:.4f}')
    
    print(classification_report(pt_labels, pt_preds, labels=ALL_LABELS,
                                target_names=STAGE_NAMES, zero_division=0))
    
    for norm, sfx in [(None, 'raw'), ('true', 'normalized')]:
        cm = confusion_matrix(pt_labels, pt_preds, labels=ALL_LABELS, normalize=norm)
        fig, ax = plt.subplots(figsize=(8, 6))
        fmt = '.2f' if norm else 'd'
        ConfusionMatrixDisplay(cm, display_labels=STAGE_NAMES).plot(
            ax=ax, cmap='Blues', values_format=fmt)
        ax.set_title(f'Patient ({method}) CM ({sfx})')
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'patient_{method}_cm_{sfx}.png'), dpi=150)
        plt.close(fig)
    
    plot_roc(pt_labels, pt_probs, STAGE_NAMES,
             os.path.join(FIGURES_DIR, f'patient_{method}_roc.png'))

In [ ]:
# ═════════════════════════════════════════════════════════════════
# GRAD-CAM
# ═════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('GRAD-CAM')
print('='*60)

def load_img(row, base, sz=(224,224)):
    img = Image.open(os.path.join(base, row['path_image'])).convert('L')
    img = img.resize((sz[1], sz[0]))
    arr = np.array(img, dtype=np.float32) / 255.0
    return np.stack([arr]*3, axis=-1)

np.random.seed(42)
n_gc = 5

for tag, mask in [('correct', y_pred == y_true),
                   ('incorrect', y_pred != y_true)]:
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        print(f'No {tag} predictions')
        continue
    if len(idxs) > n_gc:
        idxs = np.random.choice(idxs, n_gc, replace=False)
    for i, idx in enumerate(idxs):
        row = test_df.iloc[idx]
        img = load_img(row, DATA_DIR, IMG_SIZE)
        try:
            hm = generate_gradcam(model, img)
            title = (f'{tag.upper()}: True={STAGE_NAMES[y_true[idx]]}, '
                     f'Pred={STAGE_NAMES[y_pred[idx]]}')
            save_gradcam_overlay(img, hm,
                os.path.join(GRADCAM_DIR, f'{tag}_{i}.png'), title=title)
        except Exception as e:
            print(f'Grad-CAM error ({tag} {i}): {e}')

print(f'Saved to {GRADCAM_DIR}/')

In [ ]:
# ── Gallery ───────────────────────────────────────────────────────
gc = sorted([f for f in os.listdir(GRADCAM_DIR) if f.endswith('.png')])
if gc:
    n = min(len(gc), 10)
    cols = min(5, n); rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = np.array(axes).flatten() if n > 1 else [axes]
    for i, f in enumerate(gc[:n]):
        axes[i].imshow(Image.open(os.path.join(GRADCAM_DIR, f)))
        axes[i].set_title(f.replace('.png',''), fontsize=9)
        axes[i].axis('off')
    for i in range(n, len(axes)): axes[i].axis('off')
    plt.suptitle('Grad-CAM Gallery', fontsize=16)
    plt.tight_layout(); plt.show()

print('\n✅ DONE')